# Harrow-Hassidim-Lloyd (HHL) Algorithm

**Download Notebook** - {nb-download}`hhl.ipynb`

An end-to-end example of solving linear systems of equations using the HHL algorithm in Guppy.

## Mathematical Background

The **HHL algorithm** (Harrow, Hassidim, Lloyd, 2009) solves linear systems of equations of the form:

$$A \vec{x} = \vec{b} \iff \vec{x} = A^{-1} \vec{b},$$

where $A$ is a Hermitian $N \times N$ matrix ($N = 2^{n}$) and $\vec{b}$ is a normalized vector represented by the quantum state $\ket{b} = \sum_{j} \beta_j \ket{u_j}$, expressed in the orthonormal eigenbasis $\{\ket{u_j}\}$ of $A$ with eigenvalues $\{\lambda_j\}$:

$$A \ket{u_j} = \lambda_j \ket{u_j}.$$

The goal of the algorithm is to prepare the normalized quantum state:

$$\ket{x} \propto A^{-1} \ket{b} = \sum_{j} \frac{\beta_j}{\lambda_j} \ket{u_j}.$$

### Algorithmic Workflow

1. **State Preparation**: Prepare the input state $\ket{b}$ on the system register and initialize an $n_{\text{qpe}}$-qubit clock register in uniform superposition via Hadamard gates:
   $$\ket{\psi_0} = \frac{1}{\sqrt{2^{n_{\text{qpe}}}}} \sum_{k=0}^{2^{n_{\text{qpe}}}-1} \ket{k} \otimes \sum_{j} \beta_j \ket{u_j}.$$

2. **Quantum Phase Estimation (QPE)**: Evolve the system under controlled Hamiltonian simulation $e^{-i A t}$ using powers of Trotterized steps. QPE correlates the clock register with the eigenvalues $\lambda_j$:
   $$\ket{\psi_1} = \sum_{j} \beta_j \ket{\tilde{\lambda}_j}_{\text{clock}} \ket{u_j}_{\text{system}}.$$

3. **Controlled Ancilla Rotation**: Rotate an auxiliary ancilla qubit initialized in $\ket{0}$ by an angle conditioned on the estimated eigenvalue $\tilde{\lambda}_j$:
   $$\ket{\psi_2} = \sum_{j} \beta_j \ket{\tilde{\lambda}_j} \ket{u_j} \left( \sqrt{1 - \frac{C^2}{\tilde{\lambda}_j^2}} \ket{0}_{\text{ancilla}} + \frac{C}{\tilde{\lambda}_j} \ket{1}_{\text{ancilla}} \right),$$
   where $C$ is a scaling factor chosen such that $C / |\tilde{\lambda}_j| \leq 1$ for all eigenvalues.

4. **Inverse Quantum Phase Estimation (IQPE)**: Uncompute the clock register with inverse QPE followed by Hadamards, restoring the clock register to $\ket{0}$:
   $$\ket{\psi_3} = \sum_{j} \beta_j \ket{0}_{\text{clock}} \ket{u_j}_{\text{system}} \left( \sqrt{1 - \frac{C^2}{\tilde{\lambda}_j^2}} \ket{0}_{\text{ancilla}} + \frac{C}{\tilde{\lambda}_j} \ket{1}_{\text{ancilla}} \right).$$

5. **Repeat-Until-Success (RUS)**: Measure the ancilla qubit. Upon observing outcome $\ket{1}$, the clock register is discarded at $\ket{0}$ and the system register is left in the desired state:
   $$\ket{x} \propto \sum_{j} \frac{\beta_j}{\lambda_j} \ket{u_j} = A^{-1} \ket{b}.$$

In [6]:
import numpy as np
import zixy.qubit.pauli as zqp
from guppylang import guppy
from guppylang.std.builtins import array
from guppylang.std.debug import state_output
from guppylang.std.quantum import discard_array, h, measure, qubit
from selene_sim import Quest
from typing import no_type_check

from guppyalgos.algorithms.linear_systems import (
    create_eigenvalue_inversion,
    hhl,
)
from guppyalgos.algorithms.time_evolution.trotter import (
    cntrl_ham_sim_trotter,
    cntrl_trotter_first_order,
)
from guppyalgos.algorithms.time_evolution.trotter.trotter_sequence import (
    cntrl_trotter_from_sequence,
)
from guppyalgos.primitives.measurement import discard_array_zero
from guppyalgos.utils import qarray

## Implementing HHL in Guppy

We provide HHL with two independent Guppy functions:
- `controlled_hamiltonian_simulation`: controlled $e^{iAt}$ for any signed integer multiple of the configured time step. Positive powers perform QPE and negative powers uncompute it.
- `eigenvalue_inversion`: the clock-conditioned ancilla rotation implementing the reciprocal eigenvalue amplitudes.

The notebook builds the controlled simulation directly from first-order Trotter primitives and the eigenvalue inversion from a multiplexed rotation. HHL itself does not depend on either implementation. We embed the resulting routines inside a **Repeat-Until-Success (RUS)** loop that repeats state preparation and HHL until the ancilla measurement yields `1`.

In [2]:
# Pauli Hamiltonian operator for A
ham_op = zqp.RealTermSum.from_str("(1.5, I0 I1), (0.5, X0 X1)")
A_matrix = ham_op.to_sparse_matrix(False).toarray()

# Input state |b> = |+0>
b_vector = np.array([1.0, 1.0, 0.0, 0.0], dtype=np.complex128) / np.sqrt(2)

# Classical exact solution A^-1 |b>
expected_x = np.linalg.solve(A_matrix, b_vector)
expected_x_normalized = expected_x / np.linalg.norm(expected_x)

print(f"Matrix A (4x4 non-diagonal):\n{A_matrix}")
print(f"\nInput state |b>:\n{b_vector}")
print(f"\nExpected solution state |x> (normalized):\n{expected_x_normalized}")

Matrix A (4x4 non-diagonal):
[[1.5+0.j 0. +0.j 0. +0.j 0.5+0.j]
 [0. +0.j 1.5+0.j 0.5+0.j 0. +0.j]
 [0. +0.j 0.5+0.j 1.5+0.j 0. +0.j]
 [0.5+0.j 0. +0.j 0. +0.j 1.5+0.j]]

Input state |b>:
[0.70710678+0.j 0.70710678+0.j 0.        +0.j 0.        +0.j]

Expected solution state |x> (normalized):
[ 0.67082039+0.j  0.67082039+0.j -0.2236068 +0.j -0.2236068 +0.j]


## Implementing HHL in Guppy

We provide HHL with two independent Guppy functions:
- `controlled_hamiltonian_simulation`: controlled $e^{iAt}$ for any signed integer multiple of the configured time step. Positive powers perform QPE and negative powers uncompute it.
- `eigenvalue_inversion`: the clock-conditioned ancilla rotation implementing the reciprocal eigenvalue amplitudes.

The notebook builds the controlled simulation directly from first-order Trotter primitives and the eigenvalue inversion from a multiplexed rotation. HHL itself does not depend on either implementation. We embed the resulting routines inside a **Repeat-Until-Success (RUS)** loop that repeats state preparation and HHL until the ancilla measurement yields `1`.

In [7]:
n_qpe = 3
time_step = -0.5
rotation_scalar = 1.0

# State preparation function for |b> = |+0>
@guppy
def prepare_b(qs: array[qubit, 2]) -> None:
    h(qs[0])

controlled_trotter_step = cntrl_trotter_first_order(ham_op, n_state_qubits=2)
ham_terms = list(ham_op.to_terms())
inverse_trotter_step = cntrl_trotter_from_sequence(
    ham_terms,
    [(term_index, 1.0) for term_index in reversed(range(len(ham_terms)))],
    2,
)
forward_simulation = cntrl_ham_sim_trotter(controlled_trotter_step, 1, time_step, 2)
inverse_simulation = cntrl_ham_sim_trotter(inverse_trotter_step, 1, -time_step, 2)

@guppy
@no_type_check
def controlled_hamiltonian_simulation(
    control: qubit,
    state_register: array[qubit, 2],
    power: int,
) -> None:
    if power >= 0:
        for _ in range(power):
            forward_simulation(control, state_register)
    else:
        for _ in range(-power):
            inverse_simulation(control, state_register)

eigenvalue_inversion = create_eigenvalue_inversion(3, rotation_scalar)


# Repeat-until-success wrapper
@guppy
@no_type_check
def run_hhl_rus() -> None:
    while True:
        qs = qarray(2)
        prepare_b(qs)
        clock_reg = qarray(3)
        ancilla = qubit()
        hhl(
            qs,
            clock_reg,
            ancilla,
            controlled_hamiltonian_simulation,
            eigenvalue_inversion,
        )
        success = measure(ancilla).read()
        if success:
            state_output("solution", qs)
            discard_array_zero(clock_reg)
            discard_array(qs)
            break
        discard_array_zero(clock_reg)
        discard_array(qs)

## Emulating and Validating the Solution

We emulate `run_hhl_rus` and extract the recorded state vector distribution for `"solution"`.

In [8]:
def switch_endianness(vec: np.ndarray) -> np.ndarray:
    """Convert simulator state vector from big-endian to little-endian convention."""
    n = int(np.log2(vec.size))
    indices = np.arange(vec.size, dtype=np.uint64)
    reversed_indices = np.zeros_like(indices)
    for _ in range(n):
        reversed_indices = (reversed_indices << 1) | (indices & 1)
        indices >>= 1
    return np.take(vec, reversed_indices)


sim_result = run_hhl_rus.emulator(n_qubits=8).run()
states = Quest.extract_states_dict(sim_result.results[0].entries)
raw_state = states["solution"].get_state_vector_distribution()[0].state
actual_state = switch_endianness(raw_state)

print(f"Actual quantum state |x>:   {actual_state}")
print(f"Expected theoretical |x>:   {expected_x_normalized}")
overlap = abs(np.vdot(actual_state, expected_x_normalized)) ** 2
print(f"Fidelity / Overlap:         {overlap:.6f}")

Actual quantum state |x>:   [ 0.67082039+0.j  0.67082039+0.j -0.2236068 +0.j -0.2236068 +0.j]
Expected theoretical |x>:   [ 0.67082039+0.j  0.67082039+0.j -0.2236068 +0.j -0.2236068 +0.j]
Fidelity / Overlap:         1.000000
